<a href="https://colab.research.google.com/github/Ratchaphon1997/Master-project/blob/main/Masterproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import transformers
from transformers import AutoModel, AutoTokenizer

# Set seed for reproducibility
import random
seed_value = 2042
random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)

# specify GPU
device = torch.device("cuda")

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/My Drive/IScode_dataset/

Mounted at /content/drive/
/content/drive/My Drive/IScode_dataset


In [ ]:
df = pd.read_csv("Data_complete_edit.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Data_complete_edit.csv'

In [ ]:
df.info()

In [ ]:
df.drop(columns=['user_name', 'user_color'], inplace=True)

In [ ]:
df[df["labels"] == 'hatespeech'].sample(5)

In [ ]:
df[df["labels"] == 'offensive language'].sample(5)

In [ ]:
df[df["labels"] == 'neither'].sample(5)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming 'labels' is a categorical column
label_counts = df['labels'].value_counts()  # Count occurrences of each label

# Create a bar chart to visualize the frequency
plt.figure(figsize=(8, 4))  # Set figure size
label_counts.plot(kind='bar', color='skyblue')  # Plot bar chart with color
plt.xlabel('Labels')  # Label for x-axis
plt.ylabel('Frequency')  # Label for y-axis
plt.title('Frequency of Labels')  # Chart title
plt.gca().spines[['top', 'right']].set_visible(False)  # Remove top and right spines

# Optional customizations (adjust as needed)
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability
plt.tight_layout()  # Adjust spacing between elements

plt.show()  # Display the plot

In [ ]:
'''# Create a dictionary mapping labels to numerical values
label_mapping = {'hatespeech': 0, 'offensive language': 1, 'neither': 2}

# Replace labels with numerical values using 'map'
df['labels'] = df['labels'].map(label_mapping)'''

# Cleansing Data & Prepocessing

In [ ]:
!pip install pythainlp
!pip install epitran
!pip install nltk
import nltk
nltk.download('all', halt_on_error=False)

In [ ]:
import re
from pythainlp.corpus import thai_stopwords
from pythainlp.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from pythainlp.util import normalize
from pythainlp.util import num_to_thaiword
stopwords = thai_stopwords()

def remove_non_thai(text):
    """Remove non-Thai language characters from the text."""
    thai_pattern = re.compile(r'[\u0E00-\u0E7F\s]+')  # Thai Unicode range
    return ''.join(thai_pattern.findall(text))


def remove_alphanumerics_th(text, replacement_text=" "):
    """Remove alphanumerics from string but leave single quote be for Thai text."""
    pattern = re.compile(r"[^ก-๙a-zA-Z0-9']+")
    return pattern.sub(replacement_text, text)


def remove_urls_th(text, replacement_text=""):
    """Remove URLs from string for Thai text."""
    pattern = re.compile(r"https?://\S+|www\.\S+")
    return pattern.sub(replacement_text, text)


def remove_twitter_handles_th(text, replacement_text=""):
    """Remove twitter handles from string for Thai text."""
    pattern = re.compile(r"@[\w]+")
    return pattern.sub(replacement_text, text)



def remove_multiple_whitespaces_th(text, replacement_text=" "):
    """Remove multiple whitespaces from string for Thai text."""
    pattern = re.compile(r"\s{2,}")
    return pattern.sub(replacement_text, text)


def decode_html_character_references(text):
    """Decode HTML chacarters in string, e.g. &#38; and &amp;."""
    import html
    return html.unescape(text)

# Remove numbers
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def clean_html_tags(text):
    """Remove HTML tags from text."""
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)


def remove_empty_brackets(text):
    """Remove empty brackets such as (), {}, []."""
    return re.sub(r'\(\)|\{\}|\[\]', '', text)


def remove_repeated_characters(text):
    """Remove repeated characters exceeding 3 characters, e.g., 'น้อนนน' to 'น้อน'."""
    return re.sub(r'(.)\1{3,}', r'\1\1\1', text)


def remove_duplicate_words(text):
    """Remove duplicate consecutive words, e.g., 'อะอะอะ' to 'อะอะ'."""
    words = text.split()
    unique_words = [words[i] for i in range(len(words)) if i == 0 or words[i] != words[i-1]]
    return ' '.join(unique_words)

# Remove spaces at the beginning and end of the tweet
def remove_spaces_tweets(text):
    return text.strip()
def remove_thai_stopwords(text):
    """Remove Thai stopwords from the text."""
    words = text.split()
    filtered_words = [word for word in words if word not in stopwords]
    return ' '.join(filtered_words)


def clean_text(text):
    """Combines all preprocessing steps for Thai text."""
    text = decode_html_character_references(text)
    text = remove_non_thai(text)
    text = remove_twitter_handles_th(text)
    text = remove_numbers(text)
    text = remove_empty_brackets(text)
    text = remove_urls_th(text)
    text = remove_repeated_characters(text)
    text = remove_alphanumerics_th(text)
    text = remove_multiple_whitespaces_th(text)
    text = remove_duplicate_words(text)
    text = normalize(text)
    text = remove_thai_stopwords(text)
    text = text.strip()  # Remove leading/trailing whitespace
    # Check if the text is empty after cleaning
   # Check if the text is empty or None after cleaning
    if not text or text == 'None':
        text = None  # Replace empty string or 'None' with None

    return text


df = (df
      .assign(clean_message=lambda df_: df_["message"].apply(clean_text))
)

df.sample(10)

In [ ]:
# Count missing values in each column
missing_values_per_column = df.isna().sum()

# Print the missing values count
print(missing_values_per_column)

## Exploratory Data Analysis¶
Counts and Lenght:
Start by checking how long the reviews ar-

Character co- unt
Word c- ount
Mean word l- ength
Mean sentence length

In [ ]:
# check class distribution
df['labels'].value_counts(normalize = True)

In [ ]:
print(f'There are around {int(df["clean_message"].duplicated().sum())} duplicated tweets, we will remove them.')

In [ ]:
df.drop_duplicates("clean_message", inplace=True)

In [ ]:
df.dropna()

In [ ]:
# Count missing values in each column
missing_values_per_column = df.isna().sum()

# Print the missing values count
print(missing_values_per_column)

# Show rows with any 'None' values (using boolean indexing)
rows_with_none = df[df.isna().any(axis=1)]

# Drop rows with any 'None' values (modifying the original DataFrame)
df.drop(rows_with_none.index, inplace=True)


In [ ]:
# check class distribution
df['labels'].value_counts(normalize = True)


In [ ]:
df['text_len'] = [len(text.split()) for text in df.clean_message]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(7,5))
ax = sns.countplot(x='text_len', data=df[df['text_len']<10], palette='mako')
plt.title('Count of message with less than 10 words', fontsize=20)
plt.yticks([])
ax.bar_label(ax.containers[0])
plt.ylabel('count')
plt.xlabel('')
plt.show()

In [ ]:
df.sort_values(by=['text_len'], ascending=True)

In [ ]:
plt.figure(figsize=(10,5))
ax = sns.countplot(x='text_len', data=df[(df['text_len']<=1000) & (df['text_len'] > 0)], palette='Blues_r')
plt.title('Count of Message with high number of words', fontsize=25)
plt.yticks([])
ax.bar_label(ax.containers[0])
plt.ylabel('count')
plt.xlabel('')
plt.show()

In [ ]:
max_len = np.max(df['text_len'])
max_len

In [ ]:
df.sort_values(by=["text_len"], ascending=False)

In [ ]:
print(len(df['clean_message']))

In [ ]:
# Create a dictionary mapping labels to numerical values
label_mapping = {'hatespeech': 0, 'offensive language': 1, 'neither': 2}

# Replace labels with numerical values using 'map'
df['labels'] = df['labels'].map(label_mapping)

In [ ]:
X = df['clean_message']
y = df['labels']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed_value)

# Train - Validation split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.3, stratify=y_train, random_state=seed_value)

In [ ]:
print("Number of samples in X_train:", len(X_train))
print("Number of samples in X_test:", len(X_test))
print("Number of samples in X_valid:", len(X_valid))

In [ ]:
print(X_test)

In [ ]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler()
X_train_os, y_train_os = ros.fit_resample(np.array(X_train).reshape(-1,1),np.array(y_train).reshape(-1,1))

In [ ]:
# Layer ‘Flatten’ นั้นมีไว้เพื่อแปลงข้อมูลจากภาพหลาย channel ให้เป็นเวคเตอร์ ที่เราสามารถส่งต่อให้ชั้น MLP มาตรฐานต่อได้
X_train_os = X_train_os.flatten()
y_train_os = y_train_os.flatten()

In [ ]:
(unique, counts) = np.unique(y_train_os, return_counts=True)
np.asarray((unique, counts)).T

# BERT Tokenization

In [ ]:
model_ckpt = "FacebookAI/xlm-roberta-base"

In [ ]:
# import BERT-base pretrained model
bert = AutoModel.from_pretrained(model_ckpt , num_labels = 3)

# Load the BERT tokenizer
#-Tokenizing (splitting strings in sub-word token strings), converting tokens strings to ids and back, and encoding/decoding (i.e., tokenizing and converting to integers).
#-Adding new tokens to the vocabulary in a way that is independent of the underlying structure (BPE, SentencePiece…).
#-Managing special tokens
tokenizer = AutoTokenizer.from_pretrained(model_ckpt,do_lower_case=True)

In [ ]:
MAX_LEN = 128

In [ ]:
def bert_tokenizer(data):
    input_ids = []
    attention_masks = []
    for sent in data:
        encoded_sent = tokenizer.encode_plus(
            text=sent,
            add_special_tokens=True,        # Add `[CLS]` and `[SEP]` special tokens
            max_length=MAX_LEN,             # Choose max length to truncate/pad
            pad_to_max_length=True,         # Pad sentence to max length
            return_attention_mask=True      # Return attention mask
            )
        input_ids.append(encoded_sent.get('input_ids'))
        attention_masks.append(encoded_sent.get('attention_mask'))

    # Convert lists to tensors
    input_ids = torch.tensor(input_ids)
    attention_masks = torch.tensor(attention_masks)

    return input_ids, attention_masks

In [ ]:
# Tokenize train tweets
encoded_tweets = [tokenizer.encode(sent, add_special_tokens=True) for sent in X_train]

# Find the longest tokenized tweet
max_len = max([len(sent) for sent in encoded_tweets])
print('Max length: ', max_len)

## Example Tokenizer

In [ ]:
text = "สวยมากครับลูกพรี่"

print(tokenizer(text))
tokens = tokenizer.tokenize(text)

print("\n-------------------------------------------------\n",tokens)

In [ ]:
train_inputs, train_masks = bert_tokenizer(X_train_os)
val_inputs, val_masks = bert_tokenizer(X_valid)
test_inputs, test_masks = bert_tokenizer(X_test)

# Data preprocessing for PyTorch BERT model

In [ ]:
# Convert target columns to pytorch tensors format
train_labels = torch.from_numpy(y_train_os)
# Convert the Series to a NumPy array
val_labels = torch.from_numpy(y_valid.to_numpy())
test_labels = torch.from_numpy(y_test.to_numpy())

In [ ]:
batch_size = 32

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
# Create the DataLoader for our training set
train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

# Create the DataLoader for our validation set
val_data = TensorDataset(val_inputs, val_masks, val_labels)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler=val_sampler, batch_size=batch_size)

# Create the DataLoader for our test set
test_data = TensorDataset(test_inputs, test_masks, test_labels)
test_sampler = SequentialSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)

In [ ]:
class Bert_Classifier(nn.Module):
    def __init__(self, freeze_bert=False):
        super(Bert_Classifier, self).__init__()
        # Specify hidden size of BERT, hidden size of the classifier, and number of labels
        n_input = 768
        n_hidden = 50
        n_output = 3

        # Instantiate BERT model
        self.bert = bert

        # Instantiate the classifier (a fully connected layer followed by a ReLU activation and another fully connected layer)
        self.classifier = nn.Sequential(
            nn.Linear(n_input, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_output)
        )

        # Freeze the BERT model weights if freeze_bert is True (useful for feature extraction without fine-tuning)
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        # Feed input data (input_ids and attention_mask) to BERT
        outputs = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask)

        # Extract the last hidden state of the `[CLS]` token from the BERT output (useful for classification tasks)
        last_hidden_state_cls = outputs[0][:, 0, :]

        # Feed the extracted hidden state to the classifier to compute logits
        logits = self.classifier(last_hidden_state_cls)

        return logits

In [ ]:
from transformers import AdamW, get_linear_schedule_with_warmup
# Function for initializing the BERT Classifier model, optimizer, and learning rate scheduler
def initialize_model(epochs=5):
    # Instantiate Bert Classifier
    model_bert = Bert_Classifier(freeze_bert=False)

    model_bert.to(device)

    # Set up optimizer
    optimizer = AdamW(model_bert.parameters(),
                      lr=5e-5,    # learning rate, set to default value
                      eps=1e-8    # decay, set to default value
                      )

    # Calculate total number of training steps
    total_steps = len(train_dataloader) * epochs

    # Define the learning rate scheduler
    scheduler = get_linear_schedule_with_warmup(optimizer,
                                                num_warmup_steps=0, # Default value
                                                num_training_steps=total_steps)
    return model_bert, optimizer, scheduler

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS=10

In [ ]:
print(device)

In [ ]:
model_bert, optimizer, scheduler = initialize_model(epochs=EPOCHS)

In [ ]:
print(model_bert)

# BERT Training

In [ ]:
# Define Cross entropy Loss function for the multiclass classification task
loss_fn = nn.CrossEntropyLoss()
def bert_train(model, train_dataloader, val_dataloader=None, epochs=4, evaluation=False):
    train_loss_values = []
    val_loss_values = []
    val_accuracy_values = []

    print("Start training...\n")
    for epoch_i in range(epochs):
        print("-"*10)
        print("Epoch : {}".format(epoch_i+1))
        print("-"*10)
        print("-"*38)
        print(f"{'BATCH NO.':^7} | {'TRAIN LOSS':^12} | {'ELAPSED (s)':^9}")
        print("-"*38)

        # Measure the elapsed time of each epoch
        t0_epoch, t0_batch = time.time(), time.time()

        # Reset tracking variables at the beginning of each epoch
        total_loss, batch_loss, batch_counts = 0, 0, 0

        ###TRAINING###

        # Put the model into the training mode
        model.train()

        for step, batch in enumerate(train_dataloader):
            batch_counts +=1

            b_input_ids, b_attn_mask, b_labels = tuple(t.to(device) for t in batch)

            # Zero out any previously calculated gradients
            model.zero_grad()

            # Perform a forward pass and get logits.
            logits = model(b_input_ids, b_attn_mask)

            # Compute loss and accumulate the loss values
            loss = loss_fn(logits, b_labels)
            batch_loss += loss.item()
            total_loss += loss.item()

            # Perform a backward pass to calculate gradients
            loss.backward()

            # Clip the norm of the gradients to 1.0 to prevent "exploding gradients"
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Update model parameters:
            # fine tune BERT params and train additional dense layers
            optimizer.step()
            # update learning rate
            scheduler.step()

            # Print the loss values and time elapsed for every 100 batches
            if (step % 100 == 0 and step != 0) or (step == len(train_dataloader) - 1):
                # Calculate time elapsed for 20 batches
                time_elapsed = time.time() - t0_batch

                print(f"{step:^9} | {batch_loss / batch_counts:^12.6f} | {time_elapsed:^9.2f}")

                # Reset batch tracking variables
                batch_loss, batch_counts = 0, 0
                t0_batch = time.time()

        # Calculate the average loss over the entire training data
        avg_train_loss = total_loss / len(train_dataloader)

        # Append training loss to the train_loss_values list
        train_loss_values.append(avg_train_loss)

        ###EVALUATION###

        # Put the model into the evaluation mode
        model.eval()

        # Define empty lists to host accuracy and validation for each batch
        val_accuracy = []
        val_loss = []

        for batch in val_dataloader:
            batch_input_ids, batch_attention_mask, batch_labels = tuple(t.to(device) for t in batch)

            # We do not want to update the params during the evaluation,
            # So we specify that we dont want to compute the gradients of the tensors
            # by calling the torch.no_grad() method
            with torch.no_grad():
                logits = model(batch_input_ids, batch_attention_mask)

            loss = loss_fn(logits, batch_labels)

            val_loss.append(loss.item())

            # Get the predictions starting from the logits (get index of highest logit)
            preds = torch.argmax(logits, dim=1).flatten()

            # Calculate the validation accuracy
            accuracy = (preds == batch_labels).cpu().numpy().mean() * 100
            val_accuracy.append(accuracy)

        # Compute the average accuracy and loss over the validation set
        val_loss = np.mean(val_loss)
        val_accuracy = np.mean(val_accuracy)

        # Append validation loss and accuracy to the respective lists
        val_loss_values.append(val_loss)
        val_accuracy_values.append(val_accuracy)

        # Print performance over the entire training data
        time_elapsed = time.time() - t0_epoch
        print("-"*61)
        print(f"{'AVG TRAIN LOSS':^12} | {'VAL LOSS':^10} | {'VAL ACCURACY (%)':^9} | {'ELAPSED (s)':^9}")
        print("-"*61)
        print(f"{avg_train_loss:^14.6f} | {val_loss:^10.6f} | {val_accuracy:^17.2f} | {time_elapsed:^9.2f}")
        print("-"*61)
        print("\n")

    print("Training complete!")

    return train_loss_values, val_loss_values, val_accuracy_values

# Assuming you have defined and initialized your model, optimizer, scheduler, and dataloaders

# Train the model and collect loss values
train_loss_values, val_loss_values, val_accuracy_values = bert_train(model_bert, train_dataloader, val_dataloader, epochs=10 , evaluation=True)

In [ ]:
# Plot learning curve
epochs = range(1, len(train_loss_values) + 1)
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_loss_values, 'b-o', label='Training Loss')
plt.plot(epochs, val_loss_values, 'r-o', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Create a dictionary with loss values
loss_data = {
    'Epoch': range(1, len(train_loss_values) + 1),
    'Accuracy' : val_accuracy_values,
    'Training Loss': train_loss_values,
    'Validation Loss': val_loss_values
}

# Create DataFrame from the dictionary
loss_df = pd.DataFrame(loss_data)

In [ ]:
# Calculate means and standard deviations
accuracy_mean = loss_df['Accuracy'].mean()
accuracy_std = loss_df['Accuracy'].std()

training_loss_mean = loss_df['Training Loss'].mean()
training_loss_std = loss_df['Training Loss'].std()

validation_loss_mean = loss_df['Validation Loss'].mean()
validation_loss_std = loss_df['Validation Loss'].std()

# Print descriptive statistics
print(loss_df.describe(include='all'))  # Include all data types (object and numerics)

# Print means and standard deviations with informative labels
print("\n**Accuracy Statistics**")
print(f"Mean: {accuracy_mean:.4f}")
print(f"Standard Deviation: {accuracy_std:.4f}")

print("\n**Training Loss Statistics**")
print(f"Mean: {training_loss_mean:.4f}")
print(f"Standard Deviation: {training_loss_std:.4f}")

print("\n**Validation Loss Statistics**")
print(f"Mean: {validation_loss_mean:.4f}")
print(f"Standard Deviation: {validation_loss_std:.4f}")

# Print the DataFrame (optional)
print("\n\nLoss DataFrame:\n", loss_df)

In [ ]:
torch.save(model_bert, 'wangchanberta-base-att-spm-uncased-sentiment.pt')

In [ ]:
def bert_predict(model, test_dataloader):

    # Define empty list to host the predictions
    preds_list = []

    # Put the model into evaluation mode
    model.eval()

    for batch in test_dataloader:
        batch_input_ids, batch_attention_mask = tuple(t.to(device) for t in batch)[:2]

        # Avoid gradient calculation of tensors by using "no_grad()" method
        with torch.no_grad():
            logit = model(batch_input_ids, batch_attention_mask)

        # Get index of highest logit
        pred = torch.argmax(logit,dim=1).cpu().numpy()
        # Append predicted class to list
        preds_list.extend(pred)

    return preds_list

In [ ]:
bert_preds = bert_predict(model_bert, test_dataloader)

In [ ]:
print('Classification Report for BERT :\n', classification_report(y_test, bert_preds))

In [ ]:
def conf_matrix(y, y_pred, title, labels):
    fig, ax =plt.subplots(figsize=(7.5,7.5))
    ax=sns.heatmap(confusion_matrix(y, y_pred), annot=True, cmap="Purples", fmt='g', cbar=False, annot_kws={"size":30})
    plt.title(title, fontsize=25)
    ax.xaxis.set_ticklabels(labels, fontsize=16)
    ax.yaxis.set_ticklabels(labels, fontsize=14.5)
    ax.set_ylabel('Test', fontsize=25)
    ax.set_xlabel('Predicted', fontsize=25)
    plt.show()

In [ ]:
# Print confusion matrix
conf_mat = confusion_matrix(y_test, bert_preds)
# Define class labels
class_labels = ['hatespeech', 'offensive language', 'neither']

# Plot confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(conf_mat, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(class_labels))
plt.xticks(tick_marks, class_labels, rotation=45)
plt.yticks(tick_marks, class_labels)

fmt = 'd'
thresh = conf_mat.max() / 2.
for i in range(conf_mat.shape[0]):
    for j in range(conf_mat.shape[1]):
        plt.text(j, i, format(conf_mat[i, j], fmt),
                 ha="center", va="center",
                 color="white" if conf_mat[i, j] > thresh else "black")

plt.ylabel('True labels')
plt.xlabel('Predicted labels')
plt.tight_layout()
plt.show()

# Evaluating our model’s performance

In [ ]:
import torch

def predict_sentiment(text, model, tokenizer, device, max_length=128):
    model.eval()

    # Tokenize the text
    encoding = tokenizer(text, return_tensors='pt', max_length=max_length, padding='max_length', truncation=True)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Perform forward pass
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs, dim=1)

    # Define class names
    class_names = ["hatespeech", "offensive language", "neither"]
    # Return the predicted sentiment
    return class_names[preds.item()]





In [ ]:
import torch

def probabilities_1(text, model, tokenizer, device, max_length=128):
  """Predicts sentiment for a given text and returns the probabilities for each class.

  Args:
      text: The text to predict sentiment for.
      model: The sentiment classification model.
      tokenizer: The tokenizer for processing text.
      device: The device to use for computation (CPU or GPU).
      max_length: The maximum sequence length for the model (default: 128).

  Returns:
      A dictionary containing class names and probabilities.
  """

  model.eval()

  # Tokenize the text
  encoding = tokenizer(text, return_tensors='pt', max_length=max_length, padding='max_length', truncation=True)
  input_ids = encoding['input_ids'].to(device)
  attention_mask = encoding['attention_mask'].to(device)

  # Perform forward pass
  with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs[0]  # Assuming the model outputs logits

  # Get class names and probabilities
  class_names = ["hatespeech", "offensive language", "neither"]

  # Return probabilities dictionary only
  return {name: float(logit.item()) for name, logit in zip(class_names, logits.squeeze(0))}


In [ ]:
# Define multiple review texts
review_texts = [
    "ทำไมทำอย่างงี้ครับ",
    "ชิบหายละไอสัส",
    "ทำไมทีมเราถึงไม่ชนะเลยครับ",
    "ไอเอ ไอบี ไอซี เนี้ยเล่นโง่มากไม่ไหวเลยครับ",
    "ไอสัสเอ้ยกากชิบหาย"
]

# Predict sentiment for each text
for text in review_texts:
    sentiment = predict_sentiment(text, model_bert, tokenizer, device)  # Assuming the function exists
    probabilities = probabilities_1(text, model_bert, tokenizer, device)

    # Print the review text, predicted sentiment with label, and probabilities
    print("Review Text:",text)
    print(f"\nPredicted Sentiment: {sentiment}")
    print(f"\nPredicted Sentiment Probabilities: {probabilities}\n")
    print("-----------------------------------------------------------------------------------------")